# Experimento de control 2×2 — ¿el modelo lee la letra o predice el lenguaje clínico?

**Comentario 56 del revisor.** El sintético mezcla dos cosas: *cómo se escribe* (letra EMNIST renderizada por código) y *qué se escribe* (texto clínico SPACCC). Aquí se genera el **mismo** generador, con **el mismo estilo de página y el mismo encabezado por página** (misma semilla), cambiando **solo el texto del cuerpo**, y se evalúa **sin entrenar nada** (inferencia) el Qwen2.5-VL-3B zero-shot y con el LoRA entrenado.

| Condición | Texto del cuerpo (Dx / Tto / Evolución) | Qué aísla |
|---|---|---|
| `clinico_original` | SPACCC + abreviaturas (como en el entrenamiento) | referencia (¡con la semilla del experimento, no la de entrenamiento!) |
| `clinico_barajado` | mismas palabras clínicas, **orden aleatorio** | vocabulario clínico sin sintaxis |
| `general_es` | frases generales en español moderno (no clínicas) | dominio del texto |
| `rodrigo_texto` | líneas del *train* de Rodrigo (español del s. XVI) | texto no clínico y arcaico, letra sintética |
| `pseudopalabras` | letras aleatorias, misma longitud, dígitos y signos conservados | **sin** léxico: solo lectura de letras |
| `numeros` | dígitos, dosis y signos (120/80, 3.5 mg…) | lectura de cifras, sin lenguaje |

**Cómo leer el resultado.** Si el CER del LoRA sube mucho en `pseudopalabras` y `general_es` respecto de `clinico_original`, el modelo se apoyaba en el lenguaje clínico (a priori de léxico), no solo en la letra. Las otras dos celdas del 2×2 (letra real × texto clínico) requieren una muestra real clínica (pendiente): Rodrigo ya cubre *letra real × texto no clínico*.

**Entradas Kaggle (Add Input):** `codigos-para-generacion` (`generate_synthetic_hc.py` y `emnist_to_folders.py`), `crawford/emnist`, `rodri-dataset`, `synth-lora.zip`. Internet **activado** (descarga SPACCC desde Zenodo). GPU T4 ×2. Prueba primero con `SMOKE = True`.

In [ ]:
# 0. Instalación (torchao viejo rompe peft → se desinstala ANTES de importar nada)
!pip uninstall -y -q torchao
!pip install -q rapidfuzz timm einops peft bitsandbytes
import os, gc, re, glob, json, time, random, traceback, unicodedata, zipfile
import numpy as np, pandas as pd, torch, transformers
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein
print("transformers:", transformers.__version__, "| torch:", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 1. Configuración
SMOKE = False            # True: 3 páginas por condición y 20 líneas, para probar que todo corre
SEED, MAX_NEW_TOKENS, N_WARMUP = 42, 128, 3
SEED_GEN = 2026          # distinta de la del entrenamiento (--seed 0): evita repetir páginas vistas
N_PAGES = 3 if SMOKE else 60      # ~15 líneas por página → ~900 líneas por condición
B_BOOT = 200 if SMOKE else 2000
DTYPE = torch.float16
PROMPT = ("Transcribe literalmente el texto manuscrito de esta imagen. "
          "Responde únicamente con la transcripción, sin comillas ni explicaciones.")
COND_ORDER = ["clinico_original", "pseudopalabras", "general_es", "numeros", "clinico_barajado", "rodrigo_texto"]
MODELS = {"qwen25vl3b": "zero-shot", "qwen25vl3b_lora": "LoRA"}

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA_DIR = WORK / "ctrl_data"; DATA_DIR.mkdir(exist_ok=True)
OUT = {c: WORK / f"ctrl_{c}" for c in COND_ORDER}
for p in OUT.values(): p.mkdir(exist_ok=True)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True)); return hits[0] if hits else None
GEN_PY    = find_one("/kaggle/input/**/generate_synthetic_hc.py")
EMNIST_PY = find_one("/kaggle/input/**/emnist_to_folders.py")
EMNIST_DIR= find_one("/kaggle/input/**/emnist-byclass-train.csv")
_t = find_one("/kaggle/input/**/text/transcriptions.txt")      # solo hacen falta los textos: carpeta text/ y partitions/
RODRIGO   = os.path.dirname(os.path.dirname(_t)) if _t else None
LORA_ZIP  = find_one("/kaggle/input/**/synth-lora.zip")
print("generate_synthetic_hc.py:", GEN_PY, "\nemnist_to_folders.py:", EMNIST_PY, "\nemnist csv:", EMNIST_DIR,
      "\nRodrigo:", RODRIGO, "\nsynth-lora.zip:", LORA_ZIP)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 2. Utilidades: normalización, métricas, memoria
def norm_basic(s):
    s = unicodedata.normalize("NFC", str(s))
    return re.sub(r"\s+", " ", s).strip()

def norm_strict(s):
    s = norm_basic(s).lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def clean_output(s):
    s = norm_basic(s)
    s = re.sub(r"^(transcripci[oó]n|texto|text|transcription)\s*:\s*", "", s, flags=re.I)
    return s.strip('"“”\'«» ')

def cer_wer(refs, hyps):
    ce = sum(Levenshtein.distance(r, h) for r, h in zip(refs, hyps)); ct = sum(len(r) for r in refs)
    we = sum(Levenshtein.distance(r.split(), h.split()) for r, h in zip(refs, hyps)); wt = sum(len(r.split()) for r in refs)
    return 100 * ce / max(ct, 1), 100 * we / max(wt, 1)

def compute_metrics(preds):
    refs = preds["ref"].astype(str).tolist(); hyps = preds["pred"].fillna("").astype(str).tolist()
    cer, wer = cer_wer([norm_basic(r) for r in refs], [norm_basic(h) for h in hyps])
    cer_n, wer_n = cer_wer([norm_strict(r) for r in refs], [norm_strict(h) for h in hyps])
    lat = preds["latency_s"].to_numpy()
    return {"n": len(preds), "Caracteres ref": sum(len(norm_basic(r)) for r in refs),
            "Palabras ref": sum(len(norm_basic(r).split()) for r in refs), "CER (%)": round(cer, 2), "WER (%)": round(wer, 2),
            "CER norm (%)": round(cer_n, 2), "WER norm (%)": round(wer_n, 2),
            "Latencia media (s)": round(float(lat.mean()), 3),
            "Latencia p95 (s)": round(float(np.percentile(lat, 95)), 3),
            "% alucinación": round(100 * np.mean([len(h) > 2 * len(r) + 10 for r, h in zip(refs, hyps)]), 2),
            "% vacías": round(100 * np.mean([len(h.strip()) == 0 for h in hyps]), 2),
            "Errores": int((preds["error"].fillna("") != "").sum())}

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def gpu_report(tag=""):
    for i in range(torch.cuda.device_count()):
        print(f"  [{tag}] GPU{i}: asignada {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

def reset_peaks():
    for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)

def peak_vram_gb():
    return sum(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9

## 2. Banco de letras (EMNIST) y textos (SPACCC)
Se reconstruye igual que en `data-sintetica.ipynb` (EMNIST by-class, 500 por clase).

In [ ]:
import subprocess, shutil
os.chdir(WORK)
if not (WORK / "emnist_byclass").exists():
    E = os.path.dirname(EMNIST_DIR)
    subprocess.run(["python", EMNIST_PY, "--csv", f"{E}/emnist-byclass-train.csv",
                    "--mapping", f"{E}/emnist-byclass-mapping.txt",
                    "--out", str(WORK / "emnist_byclass"), "--per_class", "500"], check=True)
if not list((WORK / "spaccc").glob("**/*.txt")):
    subprocess.run(["wget", "-q", "-O", "SPACCC.zip", "https://zenodo.org/records/2560316/files/SPACCC.zip?download=1"], check=True)
    subprocess.run(["unzip", "-q", "-o", "SPACCC.zip", "-d", "spaccc"], check=True)
CHAR_ROOT = WORK / "emnist_byclass"
TEXT_DIR = sorted((WORK / "spaccc").glob("**/corpus"))[0]
print("CHAR_ROOT:", CHAR_ROOT, "| TEXT_DIR:", TEXT_DIR, "| .txt:", len(list(TEXT_DIR.rglob("*.txt"))))

## 3. Textos de control y generación de páginas
Misma semilla en todas las condiciones → mismo estilo de página y mismo encabezado; solo cambian los campos Dx, Tto y Evolución.

In [ ]:
import importlib.util, hashlib
spec = importlib.util.spec_from_file_location("gen", GEN_PY); gen = importlib.util.module_from_spec(spec); spec.loader.exec_module(gen)
GEN_SHA1 = hashlib.sha1(open(GEN_PY, "rb").read()).hexdigest()[:12]
print("generate_synthetic_hc.py sha1:", GEN_SHA1)

GEN_SENT = [
 "Mañana tenemos reunión con el equipo a las nueve de la mañana", "La lluvia de anoche dejó las calles llenas de barro",
 "Compré pan, queso, tomates y dos botellas de agua en el mercado", "El tren sale de la estación central a las 14:30",
 "Mi hermana viaja a Cusco el próximo martes con sus amigas", "Necesito renovar el contrato de alquiler antes de fin de mes",
 "El profesor explicó el teorema con un ejemplo muy claro", "Los niños jugaron en el parque hasta que oscureció",
 "Hay que pagar la factura de luz y la de internet esta semana", "El partido terminó empatado a dos goles",
 "La biblioteca abre de lunes a sábado desde las ocho", "Preparamos arroz con pollo para el almuerzo del domingo",
 "El camino hacia la laguna es largo pero el paisaje vale la pena", "Se perdió la llave del almacén y nadie sabe dónde quedó",
 "Pedro llegó tarde porque el autobús se averió en la avenida", "Escribí una carta a mi abuela contándole sobre el viaje",
 "El concierto empieza a las siete y las entradas se agotaron", "Recuerda apagar las luces y cerrar la puerta al salir",
 "La empresa abrió una nueva sucursal en el centro de la ciudad", "Este libro cuenta la historia de un pueblo de la sierra",
 "Hoy hace mucho calor y no hay viento en toda la costa", "El jardín necesita agua dos veces por semana en verano",
 "Compramos una mesa nueva de madera para la cocina", "Los estudiantes entregaron el informe final el viernes pasado",
 "La carretera estuvo cerrada por un derrumbe cerca del puente", "Mi vecino cría gallinas y vende huevos frescos todos los días",
 "El museo exhibe pinturas del siglo diecinueve y esculturas antiguas", "Llamaré al plomero para arreglar la fuga del baño",
 "Hay veinte sillas en el salón y todas están ocupadas", "El río crece en época de lluvias y arrastra troncos y piedras",
 "Mañana temprano iremos al aeropuerto a recoger a mis tíos", "La panadería de la esquina vende los mejores panes de la zona",
 "El taller de pintura se realizará en la casa de la cultura", "No olvides traer el cuaderno, los lápices y la regla",
 "La feria del libro recibió más de mil visitantes el sábado", "El gato duerme sobre el sillón junto a la ventana",
 "Mi madre teje chompas de lana para vender en la feria", "El sábado limpiamos el patio y pintamos la reja de verde",
 "La reunión de vecinos será en el local comunal a las seis", "El cartero dejó un paquete grande en la puerta principal",
]
GEN_SHORT = ["Reunión de equipo", "Lista de compras", "Viaje a Cusco", "Pago de servicios", "Clase de pintura", "Visita al museo",
             "Cumpleaños de Ana", "Revisar el contrato", "Arreglo del baño", "Feria del libro", "Partido del sábado", "Entrega del informe"]

# vocabulario clínico (SPACCC) para medir "palabras clínicas inventadas" más adelante
_CASOS_TMP = gen.cargar_frases(TEXT_DIR, random.Random(0))
def words_strict(s): return re.findall(r"[a-zñ]+", norm_strict(s))
_cnt = {}
for caso in _CASOS_TMP:
    for fr in caso:
        for w in words_strict(fr): _cnt[w] = _cnt.get(w, 0) + 1
CLIN_VOCAB = {w for w, k in _cnt.items() if k >= 5 and len(w) >= 4}
print("Vocabulario clínico (≥5 apariciones, ≥4 letras):", len(CLIN_VOCAB))

LET = "eeeeeeeeaaaaaaaoooooosssssrrrrnnnniiiiddddllcccttuuumpbgvyqhfzjxkw"
VOW_ACC = "áéíóú"; VOW_ACC_U = "ÁÉÍÓÚ"
def pseudo_word(w, r):
    o = []
    for ch in w:
        if ch in VOW_ACC: o.append(r.choice(VOW_ACC))
        elif ch in VOW_ACC_U: o.append(r.choice(VOW_ACC_U))
        elif ch in "ñÑ": o.append(ch)
        elif ch.isalpha(): o.append(r.choice(LET).upper() if ch.isupper() else r.choice(LET))
        else: o.append(ch)               # dígitos y signos se conservan
    return "".join(o)

def ctrl_barajado(f, r):
    out = {}
    for k in ("diagnostico", "tratamiento", "evolucion"):
        w = f[k].split(); r.shuffle(w); out[k] = " ".join(w)
    return out
def ctrl_general(f, r):
    ev = ". ".join(r.sample(GEN_SENT, r.randint(2, 5)))
    if len(ev) > 650: ev = ev[:650].rsplit(" ", 1)[0]
    return {"diagnostico": r.choice(GEN_SHORT), "tratamiento": ", ".join(r.sample(GEN_SHORT, r.randint(1, 2))), "evolucion": ev}
def ctrl_pseudo(f, r):
    return {k: " ".join(pseudo_word(w, r) for w in f[k].split()) for k in ("diagnostico", "tratamiento", "evolucion")}
UNITS = ["mg", "ml", "g", "mcg", "UI", "%", "mmHg", "c/8h", "c/12h", "c/24h", "kg", "cm"]
def num_tok(r):
    p = r.random()
    if p < 0.30: return str(r.randint(1, 9999999))
    if p < 0.50: return f"{r.randint(60, 200)}/{r.randint(40, 120)}"
    if p < 0.70: return f"{r.randint(0, 99)}.{r.randint(0, 99)}"
    if p < 0.90: return f"{r.randint(1, 999)} {r.choice(UNITS)}"
    return f"{r.randint(1, 31):02d}/{r.randint(1, 12):02d}/{r.randint(2015, 2025)}"
def ctrl_numeros(f, r):
    def seq(n): return " ".join(num_tok(r) for _ in range(n))
    return {"diagnostico": seq(r.randint(2, 4)), "tratamiento": seq(r.randint(3, 6)), "evolucion": seq(r.randint(10, 24))}

ROD_LINES = []
try:
    if RODRIGO:
        def _stem(p): return os.path.splitext(os.path.basename(str(p).strip()))[0]
        def _parse(t):
            t = t.strip(); return t.replace(" ", "").replace("<space>", " ") if "<space>" in t else t
        _tr = {}
        for l in open(f"{RODRIGO}/text/transcriptions.txt", encoding="utf-8", errors="replace"):
            p = l.rstrip("\n").split(maxsplit=1)
            if len(p) == 2: _tr[_stem(p[0])] = _parse(p[1])
        _train = [_stem(l) for l in open(f"{RODRIGO}/partitions/train.txt") if l.strip()]
        ROD_LINES = [_tr[i] for i in sorted(_train) if i in _tr and len(_tr[i].strip()) > 10]
        print("Líneas de train de Rodrigo para el texto de control:", len(ROD_LINES))
except Exception as e:
    print("⚠ No se pudo leer Rodrigo (", type(e).__name__, e, ") → se omite la condición rodrigo_texto"); ROD_LINES = []

def ctrl_rodrigo(f, r):
    i = r.randrange(len(ROD_LINES) - 8); k = r.randint(3, 6)
    return {"diagnostico": ROD_LINES[r.randrange(len(ROD_LINES))][:40], "tratamiento": ROD_LINES[r.randrange(len(ROD_LINES))][:60],
            "evolucion": " ".join(ROD_LINES[i:i + k])[:650]}

CTRL = {"clinico_original": None, "clinico_barajado": ctrl_barajado, "general_es": ctrl_general,
        "rodrigo_texto": ctrl_rodrigo if ROD_LINES else "skip", "pseudopalabras": ctrl_pseudo, "numeros": ctrl_numeros}
HEADER_PREFIX = ("Nombre:", "Paciente:", "Pcte:", "N° HC:", "HC:", "N° HCl:", "Fecha:", "F:", "Edad:", "Sexo:")

def generate_condition(cond, ctrl):
    out = DATA_DIR / cond; (out / "lines").mkdir(parents=True, exist_ok=True)
    if (out / "manifest.csv").exists():
        print("⏭ ", cond, "ya generada"); return
    rng = random.Random(SEED_GEN)                    # MISMA semilla en todas → mismos estilos y encabezados
    bank = gen.GlyphBank(CHAR_ROOT, 500, rng); rnd = gen.Renderer(bank, rng)
    casos = gen.cargar_frases(TEXT_DIR, rng)
    crng = random.Random(SEED_GEN + 1)               # el texto de control usa su propio RNG
    orig = gen.make_fields
    def patched(r, frases, rn):
        f = orig(r, frases, rn)
        if ctrl:
            for k, v in ctrl(f, crng).items(): f[k] = rn.sanitize(v)
        return f
    gen.make_fields = patched
    rows, preview = [], []
    try:
        for k in range(N_PAGES):
            pid = f"hc_{k:05d}"
            for _ in range(5):
                page, text, fields, recs = gen.make_page(k, bank, rnd, casos, rng)
                if all(v in text.replace("\n", " ") for v in fields.values()): break
            for j, (t, bbox) in enumerate(recs):
                lid = f"{pid}_{j:02d}"; lp = out / "lines" / f"{lid}.png"
                crop = page.crop(bbox); crop.save(lp)
                rows.append({"id": lid, "image_path": str(lp), "text": t, "is_header": t.startswith(HEADER_PREFIX)})
                if len(preview) < 8 and not t.startswith(HEADER_PREFIX): preview.append(crop)
    finally:
        gen.make_fields = orig
    pd.DataFrame(rows).to_csv(out / "manifest.csv", index=False)
    if preview:
        w = max(p.width for p in preview); h = sum(p.height + 6 for p in preview)
        m = Image.new("RGB", (w, h), "white"); y = 0
        for p in preview: m.paste(p, (0, y)); y += p.height + 6
        m.save(OUT[cond] / "preview.png")
    print(f"✔ {cond}: {len(rows)} líneas, {N_PAGES} páginas")

for cond in COND_ORDER:
    if CTRL[cond] == "skip": print("omitida:", cond, "(no hay Rodrigo)"); continue
    generate_condition(cond, CTRL[cond])

DATA = {}
for cond in COND_ORDER:
    mf = DATA_DIR / cond / "manifest.csv"
    if mf.exists():
        d = pd.read_csv(mf, keep_default_na=False)
        d = d[d.text.str.strip().str.len() > 0]
        DATA[cond] = d.iloc[:20].reset_index(drop=True) if SMOKE else d.reset_index(drop=True)
        d.to_csv(OUT[cond] / "manifest.csv", index=False)
print({c: len(v) for c, v in DATA.items()})

In [ ]:
# Revisión visual: 8 líneas de cuerpo por condición (¡mirarlas antes de gastar horas de GPU!)
from IPython.display import display
for cond in DATA:
    pv = OUT[cond] / "preview.png"
    if pv.exists():
        print("=====", cond); display(Image.open(pv))
        print(DATA[cond][~DATA[cond].is_header].text.head(3).tolist())

In [ ]:
# 5. Adaptador LoRA
LORA_DIR = None
if LORA_ZIP:
    dst = WORK / "lora"
    if not list(dst.glob("**/adapter_config.json")):
        with zipfile.ZipFile(LORA_ZIP) as z: z.extractall(dst)
    cands = sorted(str(p) for p in dst.glob("**/adapter_config.json"))
    finales = [c for c in cands if "checkpoint-" not in c]
    LORA_DIR = os.path.dirname((finales or cands)[-1])
    cfg = json.load(open(f"{LORA_DIR}/adapter_config.json"))
    print("LORA_DIR:", LORA_DIR)
    for k in ("base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules", "task_type"):
        print(f"  {k}: {cfg.get(k)}")
    json.dump(cfg, open(WORK / "adapter_config_usado.json", "w"), indent=2)
    if "3B" not in str(cfg.get("base_model_name_or_path")):
        print("⚠️ El adaptador NO es del 3B: se omite el LoRA"); LORA_DIR = None
else:
    print("⚠️ No se encontró synth-lora.zip: se omiten las corridas LoRA")

In [ ]:
# 6. Wrappers de modelos (load / predict / unload)
class BaseOCR:
    key = name = hf_id = license = ""
    def unload(self):
        for a in ("model", "processor"):
            if hasattr(self, a): delattr(self, a)
        free_gpu()

class TrOCRWrapper(BaseOCR):
    key, name, hf_id, license = "trocr", "TrOCR-large-handwritten", "microsoft/trocr-large-handwritten", "MIT"
    def load(self):
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        self.processor = TrOCRProcessor.from_pretrained(self.hf_id)
        self.model = VisionEncoderDecoderModel.from_pretrained(self.hf_id, torch_dtype=DTYPE).to("cuda:0").eval()
    @torch.inference_mode()
    def predict(self, img):
        pv = self.processor(images=img, return_tensors="pt").pixel_values.to("cuda:0", DTYPE)
        out = self.model.generate(pv, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

class Florence2Wrapper(BaseOCR):
    key, name, hf_id, license = "florence2", "Florence-2-large", "florence-community/Florence-2-large", "MIT"
    TASK = "<OCR>"
    def load(self):
        from transformers import AutoProcessor, Florence2ForConditionalGeneration
        self.processor = AutoProcessor.from_pretrained(self.hf_id)
        self.model = Florence2ForConditionalGeneration.from_pretrained(self.hf_id, dtype=DTYPE).to("cuda:0").eval()
    @torch.inference_mode()
    def predict(self, img):
        inputs = self.processor(text=self.TASK, images=img, return_tensors="pt").to("cuda:0", DTYPE)
        ids = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
        raw = self.processor.batch_decode(ids, skip_special_tokens=False)[0]
        return self.processor.post_process_generation(raw, task=self.TASK, image_size=img.size).get(self.TASK, "")

class LlavaNextWrapper(BaseOCR):
    key, name, hf_id, license = "llava16", "LLaVA-1.6-Mistral-7B", "llava-hf/llava-v1.6-mistral-7b-hf", "Apache-2.0"
    def load(self):
        from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
        self.processor = LlavaNextProcessor.from_pretrained(self.hf_id)
        self.model = LlavaNextForConditionalGeneration.from_pretrained(
            self.hf_id, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True).eval()
    @torch.inference_mode()
    def predict(self, img):
        conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
        prompt = self.processor.apply_chat_template(conv, add_generation_prompt=True)
        inputs = self.processor(images=img, text=prompt, return_tensors="pt").to(self.model.device, DTYPE)
        out = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=self.processor.tokenizer.eos_token_id)
        return self.processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

class Qwen25VLWrapper(BaseOCR):
    key, name, hf_id, license = "qwen25vl", "Qwen2.5-VL-7B-Instruct", "Qwen/Qwen2.5-VL-7B-Instruct", "Apache-2.0"
    device_map = "auto"                                   # 7B se reparte entre las 2 T4
    def load(self):
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        self.processor = AutoProcessor.from_pretrained(self.hf_id, min_pixels=64*28*28, max_pixels=1280*28*28)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.hf_id, torch_dtype=DTYPE, device_map=self.device_map).eval()
    @torch.inference_mode()
    def predict(self, img):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
        text = self.processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], images=[img], return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        return self.processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

class Qwen25VL3BWrapper(Qwen25VLWrapper):
    key, name, hf_id = "qwen25vl3b", "Qwen2.5-VL-3B-Instruct", "Qwen/Qwen2.5-VL-3B-Instruct"
    license, device_map = "Qwen Research (no comercial)", "cuda:0"

class Qwen25VL3BLoRAWrapper(Qwen25VL3BWrapper):
    key, name = "qwen25vl3b_lora", "Qwen2.5-VL-3B-Instruct + LoRA (sintético)"
    def load(self):
        super().load()                                    # mismo processor y fp16 que el zero-shot
        from peft import PeftModel
        self.model = PeftModel.from_pretrained(self.model, LORA_DIR).merge_and_unload().eval()

class Qwen25VL7B4bitWrapper(Qwen25VLWrapper):
    """7B cuantizado a 4 bits (NF4) en UNA sola T4: la configuración del servicio del prototipo."""
    key, name = "qwen25vl7b_4bit", "Qwen2.5-VL-7B-Instruct (4 bits)"
    license, device_map = "Apache-2.0", "cuda:0"
    def load(self):
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
        self.processor = AutoProcessor.from_pretrained(self.hf_id, min_pixels=64*28*28, max_pixels=1280*28*28)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.hf_id, quantization_config=q, torch_dtype=DTYPE, device_map={"": 0}).eval()

WRAPPERS = {c.key: c for c in (TrOCRWrapper, Florence2Wrapper, LlavaNextWrapper, Qwen25VLWrapper,
                               Qwen25VL3BWrapper, Qwen25VL3BLoRAWrapper, Qwen25VL7B4bitWrapper)}

In [ ]:
# 7. Runner: guarda predicciones por línea, mide latencia y VRAM, libera memoria siempre
def run_benchmark(wrapper, df, out_dir):
    pred_path, meta_path = out_dir / f"preds_{wrapper.key}.csv", out_dir / f"meta_{wrapper.key}.json"
    if pred_path.exists() and meta_path.exists():
        print(f"⏭  {wrapper.name} ({out_dir.name}): ya evaluado"); return
    print(f"\n{'='*70}\n▶ {wrapper.name} | test: {out_dir.name} | n={len(df)}\n{'='*70}")
    rows, load_s, peak, mem_load, peak_dev = [], None, None, None, None
    free_gpu(); reset_peaks()
    baseline = sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / 1e9
    try:
        t0 = time.time(); wrapper.load(); load_s = time.time() - t0
        print(f"  Cargado en {load_s:.1f} s"); gpu_report("cargado")
        mem_load = [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())]
        for p in df["image_path"].iloc[:N_WARMUP]:
            wrapper.predict(Image.open(p).convert("RGB"))
        for r in tqdm(df.itertuples(index=False), total=len(df), desc=wrapper.name, mininterval=30):
            img = Image.open(r.image_path).convert("RGB")
            torch.cuda.synchronize(); t = time.perf_counter(); raw, err = "", ""
            try:
                raw = wrapper.predict(img)
            except torch.cuda.OutOfMemoryError:
                err = "OOM"; free_gpu()
            except Exception as e:
                err = f"{type(e).__name__}: {e}"[:200]
            torch.cuda.synchronize()
            rows.append({"id": r.id, "ref": r.text, "raw": raw, "pred": clean_output(raw),
                         "latency_s": time.perf_counter() - t, "error": err})
        peak = peak_vram_gb() - baseline
        peak_dev = [round(torch.cuda.max_memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())]
    finally:
        wrapper.unload(); gpu_report("liberado")
    if not rows:
        raise RuntimeError("sin predicciones")
    pd.DataFrame(rows).to_csv(pred_path, index=False)
    json.dump({"name": wrapper.name, "hf_id": wrapper.hf_id, "license": wrapper.license, "load_s": load_s,
               "vram_peak_gb": peak, "n": len(rows),
               "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
               "gpu_mem_after_load_gb": mem_load, "gpu_peak_gb_per_device": peak_dev,
               "n_gpus_used": int(sum(m > 0.3 for m in (mem_load or []))),
               "device_map": str(getattr(wrapper, "device_map", "cuda:0")), "transformers": transformers.__version__,
               "lora_dir": LORA_DIR if wrapper.key.endswith("_lora") else None},
              open(meta_path, "w"), indent=2)
    print(f"  ✅ {pred_path.name} | VRAM pico {peak:.2f} GB | CER {compute_metrics(pd.DataFrame(rows))['CER (%)']} %")

In [ ]:
# 6. Ejecutar: para cada condición, zero-shot y LoRA (el modelo se carga 1 vez por condición; ~1 min de carga)
LOG = []; t_total = time.time()
for cond in COND_ORDER:
    if cond not in DATA: continue
    for key in MODELS:
        if key.endswith("_lora") and not LORA_DIR: LOG.append((cond, key, "omitido: falta adaptador")); continue
        w = WRAPPERS[key]()
        try:
            run_benchmark(w, DATA[cond], OUT[cond]); LOG.append((cond, key, "ok"))
        except Exception as e:
            print(f"❌ {key} en {cond}: {type(e).__name__}: {e}"); traceback.print_exc()
            LOG.append((cond, key, f"ERROR {type(e).__name__}: {str(e)[:100]}"))
        finally:
            del w; free_gpu()
print(f"\nTiempo total: {(time.time()-t_total)/3600:.2f} h")
print(pd.DataFrame(LOG, columns=["condición", "modelo", "estado"]).to_string(index=False))

## 7. Análisis: CER por condición, IC95 por página (bootstrap) y diferencias pareadas
El remuestreo es **por página**, con el mismo índice de página en todas las condiciones (mismo estilo y encabezado), por lo que las diferencias entre condiciones son pareadas.

In [ ]:
# 7a. Matrices por página: errores de edición y caracteres de referencia
def per_line(cond, key):
    p = pd.read_csv(OUT[cond] / f"preds_{key}.csv", keep_default_na=False)
    man = pd.read_csv(OUT[cond] / "manifest.csv", keep_default_na=False)[["id", "is_header"]]
    p = p.merge(man, on="id", how="left")
    p["is_header"] = p["is_header"].astype(str).str.lower().eq("true")
    p["page"] = p["id"].str.rsplit("_", n=1).str[0]
    refs = [norm_basic(r) for r in p["ref"].astype(str)]; hyps = [norm_basic(h) for h in p["pred"].fillna("").astype(str)]
    p["e"] = [Levenshtein.distance(r, h) for r, h in zip(refs, hyps)]; p["n"] = [len(r) for r in refs]
    return p

conds = [c for c in COND_ORDER if all((OUT[c] / f"preds_{k}.csv").exists() for k in MODELS)]
LINES = {(c, k): per_line(c, k) for c in conds for k in MODELS}
pages = sorted({pg for d in LINES.values() for pg in d["page"]}); K = len(pages); pidx = {p: i for i, p in enumerate(pages)}
def mats(c, k, subset):
    d = LINES[(c, k)]; d = d if subset == "todas" else d[~d.is_header]
    E = np.zeros(K); N = np.zeros(K)
    for pg, g in d.groupby("page"): E[pidx[pg]] = g.e.sum(); N[pidx[pg]] = g.n.sum()
    return E, N
rng = np.random.default_rng(SEED); IDX = rng.integers(0, K, size=(B_BOOT, K))
def boot_cer(E, N): return 100 * E[IDX].sum(1) / np.maximum(N[IDX].sum(1), 1)
def ci(x): return f"[{np.percentile(x, 2.5):.2f}, {np.percentile(x, 97.5):.2f}]"

rows = []
for subset in ("todas", "cuerpo"):
    ref_lora = boot_cer(*mats("clinico_original", "qwen25vl3b_lora", subset)) if "clinico_original" in conds else None
    for c in conds:
        Ez, Nz = mats(c, "qwen25vl3b", subset); El, Nl = mats(c, "qwen25vl3b_lora", subset)
        bz, bl = boot_cer(Ez, Nz), boot_cer(El, Nl)
        r = {"líneas": subset, "condición": c, "n líneas": int(sum(len(LINES[(c, k)]) if subset == "todas" else (~LINES[(c, k)].is_header).sum() for k in ["qwen25vl3b"])),
             "CER zero-shot": f"{100*Ez.sum()/Nz.sum():.2f} {ci(bz)}", "CER LoRA": f"{100*El.sum()/Nl.sum():.2f} {ci(bl)}",
             "Δ LoRA−ZS": f"{100*El.sum()/Nl.sum()-100*Ez.sum()/Nz.sum():+.2f} {ci(bl-bz)}"}
        if ref_lora is not None and c != "clinico_original":
            r["LoRA vs clínico_original"] = f"{100*El.sum()/Nl.sum()-100*mats('clinico_original','qwen25vl3b_lora',subset)[0].sum()/mats('clinico_original','qwen25vl3b_lora',subset)[1].sum():+.2f} {ci(bl-ref_lora)}"
        rows.append(r)
T = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_colwidth", 60)
T.to_csv(WORK / "control_2x2_cer.csv", index=False); print(T.to_string(index=False))

In [ ]:
# 7b. WER y señales de "prior de lenguaje": palabras clínicas que el modelo escribe y que NO están en la imagen
def toks_strict(s): return re.findall(r"[a-zñ]+", norm_strict(s))
rows = []
for c in conds:
    for k, nm in MODELS.items():
        d = LINES[(c, k)]; d = d[~d.is_header]
        refs = [norm_basic(r) for r in d["ref"].astype(str)]; hyps = [norm_basic(h) for h in d["pred"].fillna("").astype(str)]
        wer = 100 * sum(Levenshtein.distance(r.split(), h.split()) for r, h in zip(refs, hyps)) / max(sum(len(r.split()) for r in refs), 1)
        inv = tot = 0
        for r, h in zip(refs, hyps):
            rs = set(toks_strict(r))
            for w in toks_strict(h):
                if len(w) >= 4:
                    tot += 1; inv += (w in CLIN_VOCAB and w not in rs)
        rows.append({"condición": c, "modelo": nm, "WER cuerpo (%)": round(wer, 2),
                     "% palabras clínicas inventadas": round(100 * inv / max(tot, 1), 2), "palabras ≥4 letras predichas": tot})
P = pd.DataFrame(rows); P.to_csv(WORK / "control_2x2_prior.csv", index=False); print(P.to_string(index=False))

# 7c. Ejemplos: 4 líneas de cuerpo por condición
ej = []
for c in conds:
    d0 = LINES[(c, "qwen25vl3b")]; d1 = LINES[(c, "qwen25vl3b_lora")].set_index("id")
    for _, r in d0[~d0.is_header].head(4).iterrows():
        ej.append({"condición": c, "ref": r["ref"][:70], "zero-shot": r["pred"][:70], "LoRA": d1.loc[r["id"], "pred"][:70]})
E = pd.DataFrame(ej); E.to_csv(WORK / "control_2x2_ejemplos.csv", index=False); print(E.to_string(index=False))

In [ ]:
# 7d. Figura: CER (líneas de cuerpo) por condición, con IC95 por página
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8.5, 4.2)); w = 0.38; xs = np.arange(len(conds))
for j, (k, nm, col) in enumerate((("qwen25vl3b", "zero-shot", "#2a6fdb"), ("qwen25vl3b_lora", "LoRA", "#d97706"))):
    pt, lo, hi = [], [], []
    for c in conds:
        E_, N_ = mats(c, k, "cuerpo"); b = boot_cer(E_, N_); p = 100 * E_.sum() / N_.sum()
        pt.append(p); lo.append(p - np.percentile(b, 2.5)); hi.append(np.percentile(b, 97.5) - p)
    ax.bar(xs + (j - .5) * w, pt, w, yerr=[lo, hi], color=col, label=nm, capsize=2, error_kw={"lw": 1})
ax.set_xticks(xs); ax.set_xticklabels([c.replace("_", "\n") for c in conds], fontsize=8); ax.set_ylabel("CER (%) en líneas de cuerpo")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=.25)
ax.set_title("Qwen2.5-VL-3B: mismo generador de letra, distinto texto", fontsize=10)
fig.tight_layout(); fig.savefig(WORK / "fig_control_2x2.png", dpi=200); plt.show()

In [ ]:
# 8. Empaquetar
json.dump({"seed_gen": SEED_GEN, "n_pages": N_PAGES, "generate_synthetic_hc_sha1": GEN_SHA1, "smoke": SMOKE,
           "lora_dir": LORA_DIR, "bootstrap_B": B_BOOT, "clin_vocab_size": len(CLIN_VOCAB), "conds": conds},
          open(WORK / "control_config.json", "w"), indent=2)
ZIP = WORK / ("control_2x2_SMOKE.zip" if SMOKE else "control_2x2.zip")
for old in WORK.glob("control_*.zip"): old.unlink()          # borra zips parciales o anteriores (liberan disco)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for c in COND_ORDER:
        for f in OUT[c].glob("*"):
            if f.is_file(): z.write(f, f"{OUT[c].name}/{f.name}")
    for f in WORK.glob("control_*"):
        if f.suffix != ".zip": z.write(f, f.name)          # nunca incluir un zip dentro del zip
    for f in WORK.glob("fig_control_*"): z.write(f, f.name)
print("Listo →", ZIP, f"({ZIP.stat().st_size/1e6:.1f} MB)")